# Milestone 2.2 — QA Pair Generation
**Input:** `extracted_papers.jsonl` from Google Drive  
**Output:** `qa_pairs.jsonl` + pushed to HuggingFace Hub as `team-name/arxiv-ml-qa`  
**Owner:** M1  
**API:** Groq API (Llama models)

Steps:
1. Install dependencies & mount Drive
2. Load extracted papers (CS.LG / CS.AI / CS.CL / CS.CV only)
3. Generate 3–4 QA pairs per abstract using Groq
4. Save as instruction-tuning format
5. Push dataset to HuggingFace Hub

## Step 0 — Install Dependencies

In [1]:
!pip install -q groq datasets tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 5.0 MB/s eta 0:00:00


## Step 1 — Mount Google Drive

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Step 2 — Configure Paths & API Keys

⚠️ **Update all three values below before running.**

In [3]:
import os

# ── UPDATE THESE ─────────────────────────────────────────────────────────────
BASE_DIR       = '/content/drive/MyDrive/arxiv-llm-project/data'
# To use Groq, please store your Groq API key in Colab secrets under the name 'GROQ_API_KEY'.
# See 'https://console.groq.com/keys' to generate a key.
# If you also plan to push to HuggingFace, store your token as 'HF_TOKEN'.
# ─────────────────────────────────────────────────────────────────────────────

INPUT_FILE  = os.path.join(BASE_DIR, 'processed', 'extracted_papers.jsonl')
OUTPUT_FILE = os.path.join(BASE_DIR, 'processed', 'qa_pairs.jsonl')
PROGRESS_FILE = os.path.join(BASE_DIR, 'processed', 'qa_progress.jsonl')  # for resume support

print('Input :', INPUT_FILE)
print('Output:', OUTPUT_FILE)
print('Input exists:', os.path.exists(INPUT_FILE))

Input : /content/drive/MyDrive/arxiv-llm-project/data/processed/extracted_papers.jsonl
Output: /content/drive/MyDrive/arxiv-llm-project/data/processed/qa_pairs.jsonl
Input exists: True


## Step 3 — Load & Filter Papers

In [4]:
import json

TARGET_CATEGORIES = {'cs.LG', 'cs.AI', 'cs.CL', 'cs.CV'}

papers = []
with open(INPUT_FILE, 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        p = json.loads(line)
        if p.get('category', '') in TARGET_CATEGORIES:
            papers.append(p)

print(f'Loaded {len(papers)} papers after category filter')
print('Category breakdown:')
from collections import Counter
cats = Counter(p.get('category', 'unknown') for p in papers)
for cat, count in cats.most_common():
    print(f'  {cat}: {count}')

Loaded 3216 papers after category filter
Category breakdown:
  cs.LG: 1710
  cs.CV: 604
  cs.CL: 535
  cs.AI: 367


## Step 4 — Set Up Groq Client

In [5]:
from groq import Groq
from google.colab import userdata

# Get the Groq API key from Colab secrets
groq_client = Groq(api_key=userdata.get('GROQ_API_KEY1'))

# Quick test
response = groq_client.chat.completions.create(
    model='llama-3.1-8b-instant',
    messages=[{'role': 'user', 'content': 'Say: API connected'}]
)
print('Groq status:', response.choices[0].message.content.strip())

Groq status: API connected


## Step 5 — Define QA Generation Prompt

Generates 3–4 QA pairs per abstract covering:
- Problem statement
- Main contribution
- Methodology
- Limitations

In [6]:
QA_PROMPT_TEMPLATE = """\
You are a research assistant helping build a training dataset for a domain-specific LLM.

Given the research paper abstract below, generate exactly 4 question-answer pairs.
Each QA pair must cover a DIFFERENT aspect:
1. Problem statement (What problem does this paper address?)
2. Main contribution (What is the key contribution or finding?)
3. Methodology (What method or approach is used?)
4. Limitations or future work (What are the limitations or future directions?)

Rules:
- Questions must be specific to THIS paper, not generic
- Answers must be 2-4 sentences, grounded only in the abstract
- Do NOT make up information not present in the abstract
- Return ONLY valid JSON, no extra text, no markdown fences

Output format (JSON array of objects):
[
  {{"question": "...", "answer": "...", "type": "problem"}},
  {{"question": "...", "answer": "...", "type": "contribution"}},
  {{"question": "...", "answer": "...", "type": "methodology"}},
  {{"question": "...", "answer": "...", "type": "limitations"}}
]

Paper Title: {title}
Abstract: {abstract}
"""

def build_prompt(paper):
    return QA_PROMPT_TEMPLATE.format(
        title=paper.get('title', 'Unknown'),
        abstract=paper.get('abstract', '')
    )

# Test prompt on first paper
print('Sample prompt preview:')
print(build_prompt(papers[0])[:600], '...')

Sample prompt preview:
You are a research assistant helping build a training dataset for a domain-specific LLM.

Given the research paper abstract below, generate exactly 4 question-answer pairs.
Each QA pair must cover a DIFFERENT aspect:
1. Problem statement (What problem does this paper address?)
2. Main contribution (What is the key contribution or finding?)
3. Methodology (What method or approach is used?)
4. Limitations or future work (What are the limitations or future directions?)

Rules:
- Questions must be specific to THIS paper, not generic
- Answers must be 2-4 sentences, grounded only in the abstract
-  ...


## Step 6 — Generate QA Pairs

- Processes all papers with **rate limiting** (15 RPM free tier)
- **Auto-resumes** from last saved position if interrupted
- Saves progress after every 10 papers

In [7]:
import itertools
import re
import json
import time
from tqdm.auto import tqdm

MODELS = itertools.cycle([
    'llama-3.1-8b-instant',
    'llama-3.3-70b-versatile'
])

def generate_qa_for_paper(paper):
    prompt = build_prompt(paper)
    try:
        response = groq_client.chat.completions.create(
            model=next(MODELS),
            messages=[{'role': 'user', 'content': prompt}]
        )
        text = response.choices[0].message.content.strip()
        qa_list = parse_qa_response(text)
        if qa_list and isinstance(qa_list, list):
            return qa_list
    except Exception as e:
        print(f'  ⚠ Error for {paper.get("paper_id", "?")}: {e}')
    return []

# ── Config ────────────────────────────────────────────────────────
REQUESTS_PER_MINUTE  = 12
SLEEP_BETWEEN_CALLS  = 5
SAVE_EVERY_N         = 10
MAX_PAPERS           = None

def parse_qa_response(text):
    """Safely parse JSON from Groq response."""
    try:
        # Strip markdown fences if present
        cleaned = re.sub(r'```json|```', '', text).strip()
        return json.loads(cleaned)
    except Exception:
        return None


# ── Resume support: load already-processed paper IDs ─────────────────────────
processed_ids = set()
all_qa_pairs  = []

if os.path.exists(PROGRESS_FILE):
    with open(PROGRESS_FILE, 'r') as f:
        for line in f:
            line = line.strip()
            if line:
                rec = json.loads(line)
                all_qa_pairs.append(rec)
                processed_ids.add(rec.get('paper_id'))
    print(f'Resuming — already have {len(all_qa_pairs)} QA pairs from {len(processed_ids)} papers')
else:
    print('Starting fresh.')

# ── Filter papers not yet processed ──────────────────────────────────────────
papers_to_process = [
    p for p in papers
    if str(p.get('paper_id', p.get('id', ''))) not in processed_ids
]
if MAX_PAPERS:
    papers_to_process = papers_to_process[:MAX_PAPERS]

print(f'Papers to process: {len(papers_to_process)}')
print(f'Estimated time   : {len(papers_to_process) * SLEEP_BETWEEN_CALLS / 60:.1f} minutes')

Resuming — already have 8439 QA pairs from 2109 papers
Papers to process: 1107
Estimated time   : 92.2 minutes


In [ ]:
# ── Main generation loop ──────────────────────────────────────────────────────
failed_papers = []

with open(PROGRESS_FILE, 'a', encoding='utf-8') as progress_f:
    for i, paper in enumerate(tqdm(papers_to_process, desc='Generating QA pairs')):
        paper_id = str(paper.get('paper_id', paper.get('id', f'paper_{i}')))

        qa_list = generate_qa_for_paper(paper)

        if not qa_list:
            failed_papers.append(paper_id)
            time.sleep(SLEEP_BETWEEN_CALLS)
            continue

        for qa in qa_list:
            record = {
                'paper_id'   : paper_id,
                'title'      : paper.get('title', ''),
                'category'   : paper.get('category', ''),
                'year'       : paper.get('year', ''),
                'url'        : paper.get('url', ''),
                'question'   : qa.get('question', ''),
                'answer'     : qa.get('answer', ''),
                'type'       : qa.get('type', ''),
                # Instruction-tuning format
                'instruction': qa.get('question', ''),
                'input'      : f"Title: {paper.get('title', '')}\nAbstract: {paper.get('abstract', '')}",
                'output'     : qa.get('answer', '')
            }
            all_qa_pairs.append(record)
            progress_f.write(json.dumps(record, ensure_ascii=False) + '\n')

        # Flush every SAVE_EVERY_N papers
        if (i + 1) % SAVE_EVERY_N == 0:
            progress_f.flush()
            tqdm.write(f'  ✓ {len(all_qa_pairs)} QA pairs so far | {len(failed_papers)} failed')

        time.sleep(SLEEP_BETWEEN_CALLS)

print(f'\n✅ Generation complete!')
print(f'   Total QA pairs : {len(all_qa_pairs)}')
print(f'   Failed papers  : {len(failed_papers)}')
if failed_papers:
    print(f'   Failed IDs     : {failed_papers[:10]}...')

Generating QA pairs:   0%|          | 0/1107 [00:00<?, ?it/s]

  ✓ 8467 QA pairs so far | 3 failed
  ✓ 8507 QA pairs so far | 3 failed
  ✓ 8547 QA pairs so far | 3 failed
  ✓ 8587 QA pairs so far | 3 failed
  ✓ 8627 QA pairs so far | 3 failed
  ✓ 8667 QA pairs so far | 3 failed
  ✓ 8707 QA pairs so far | 3 failed
  ✓ 8747 QA pairs so far | 3 failed
  ✓ 8787 QA pairs so far | 3 failed
  ✓ 8827 QA pairs so far | 3 failed
  ✓ 8867 QA pairs so far | 3 failed
  ✓ 8907 QA pairs so far | 3 failed
  ✓ 8947 QA pairs so far | 3 failed
  ✓ 9019 QA pairs so far | 5 failed
  ✓ 9059 QA pairs so far | 5 failed
  ✓ 9099 QA pairs so far | 5 failed
  ✓ 9135 QA pairs so far | 6 failed
  ✓ 9175 QA pairs so far | 6 failed


## Step 7 — Save Final qa_pairs.jsonl

In [ ]:
with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
    for record in all_qa_pairs:
        f.write(json.dumps(record, ensure_ascii=False) + '\n')

file_size_mb = os.path.getsize(OUTPUT_FILE) / (1024 * 1024)
print(f'Saved {len(all_qa_pairs)} QA pairs to:')
print(f'  {OUTPUT_FILE}')
print(f'  File size: {file_size_mb:.2f} MB')

# Sample a few
print('\n=== Sample QA Pairs ===')
for qa in all_qa_pairs[:3]:
    print(f'\n  Paper   : {qa["title"][:70]}')
    print(f'  Type    : {qa["type"]}')
    print(f'  Q       : {qa["question"]}')
    print(f'  A       : {qa["answer"][:150]}...')

## Step 8 — Quality Check

In [ ]:
from collections import Counter

# QA type distribution
type_counts = Counter(qa['type'] for qa in all_qa_pairs)
print('QA Type Distribution:')
for t, c in type_counts.most_common():
    print(f'  {t:15s}: {c}')

# Answer length stats
answer_lengths = [len(qa['answer'].split()) for qa in all_qa_pairs]
print(f'\nAnswer length (words):')
print(f'  Min    : {min(answer_lengths)}')
print(f'  Max    : {max(answer_lengths)}')
print(f'  Median : {sorted(answer_lengths)[len(answer_lengths)//2]}')

# Papers covered
papers_covered = len(set(qa['paper_id'] for qa in all_qa_pairs))
print(f'\nPapers covered : {papers_covered}')
print(f'Avg QA/paper   : {len(all_qa_pairs)/papers_covered:.1f}')

# Flag suspiciously short answers
short = [qa for qa in all_qa_pairs if len(qa['answer'].split()) < 5]
print(f'\nSuspiciously short answers (<5 words): {len(short)}')
if short:
    print('  Examples:')
    for qa in short[:3]:
        print(f'    Q: {qa["question"]}')
        print(f'    A: {qa["answer"]}')

## Step 9 — Push Dataset to HuggingFace Hub

⚠️ Make sure `HF_TOKEN` and `HF_DATASET_ID` are set in Step 2.

In [ ]:
from datasets import Dataset
from huggingface_hub import login

login(token=HF_TOKEN)

# Build HuggingFace Dataset — instruction-tuning columns only
hf_records = [
    {
        'instruction': qa['instruction'],
        'input'      : qa['input'],
        'output'     : qa['output'],
        'paper_id'   : qa['paper_id'],
        'category'   : qa['category'],
        'year'       : str(qa['year']),
        'qa_type'    : qa['type']
    }
    for qa in all_qa_pairs
    if qa['instruction'] and qa['output']  # skip empty records
]

dataset = Dataset.from_list(hf_records)
print(f'Dataset size: {len(dataset)} records')
print('Columns:', dataset.column_names)
print('\nSample record:')
print(dataset[0])

In [ ]:
# Push to Hub
dataset.push_to_hub(
    HF_DATASET_ID,
    token=HF_TOKEN,
    private=False
)

print(f'\n✅ Milestone 2.2 complete!')
print(f'   Dataset published: https://huggingface.co/datasets/{HF_DATASET_ID}')
print(f'   Total QA pairs   : {len(dataset)}')
print(f'   Ready for M2 to start fine-tuning (Milestone 2.3)')